# P03 Raw EEG Step by Step

这份 notebook 的目标不是做论文级正式复现，而是**从 raw EEG 出发，逐步感受 EEG 数据处理**。  
默认使用 `Moreira et al., 2025` 数据集里已经导出的 `P03 / ses-01 / task-phonemes` 完整 `h5` EEG 矩阵。

你会依次看到：

1. 数据和元数据长什么样
2. raw EEG 波形和频谱长什么样
3. 如何把 `events.tsv` 配成 trial 级表
4. 如何做一个最小但合理的预处理流程
5. 如何做 stimulus 对齐的 epochs
6. 如何看最基础的 ERP 条件差异
7. 如何保存中间结果

这份 notebook 故意保留了“体验式”的感觉：先摸数据，再进入简化预处理。  
因此它**不做 ICA、不做坏导插值、不做复杂统计检验**，而是把主线流程走通。

In [9]:
pip install h5py

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 1.6 MB/s  0:00:01 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path
import json
import warnings

import mne
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.unicode_minus'] = False

# =========================
# 可编辑参数区
# =========================
REPO_ROOT = Path('/Users/samxie/Research/EEG-Voice/ref_github/speech_decoding')
SIDECAR_DIR = REPO_ROOT / 'ds006104' / 'sub-P03' / 'ses-01' / 'eeg'
H5_DIR = REPO_ROOT / 'exploration_outputs' / 'edf_full_analysis' / 'sub-P03_ses-01_task-phonemes_eeg'
H5_PATH = H5_DIR / 'sub-P03_ses-01_task-phonemes_eeg_full_eeg.h5'
H5_MANIFEST_PATH = H5_DIR / 'sub-P03_ses-01_task-phonemes_eeg_full_export_manifest.json'
TMS_JSON_PATH = REPO_ROOT / 'ds006104' / 'sourcedata' / 'tms_metadata' / 'tms_parameters.json'
OUTPUT_DIR = REPO_ROOT / 'exploration_outputs' / 'notebook_play_p03_raw'

SAVE_OUTPUTS = True
RANDOM_SEED = 7
N_PREVIEW_CHANNELS = 8
PREVIEW_START_SEC = 0.0
PREVIEW_DURATION_SEC = 10.0

EPOCH_TMIN = -0.2
EPOCH_TMAX = 0.8
BASELINE = (None, -0.06)
REJECT_UV = 150.0

np.random.seed(RANDOM_SEED)

EVENTS_PATH = SIDECAR_DIR / 'sub-P03_ses-01_task-phonemes_events.tsv'
EVENTS_JSON_PATH = SIDECAR_DIR / 'sub-P03_ses-01_task-phonemes_events.json'
CHANNELS_PATH = SIDECAR_DIR / 'sub-P03_ses-01_task-phonemes_channels.tsv'
EEG_JSON_PATH = SIDECAR_DIR / 'sub-P03_ses-01_task-phonemes_eeg.json'

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_paths = {
    'H5_PATH': H5_PATH,
    'H5_MANIFEST_PATH': H5_MANIFEST_PATH,
    'EVENTS_PATH': EVENTS_PATH,
    'EVENTS_JSON_PATH': EVENTS_JSON_PATH,
    'CHANNELS_PATH': CHANNELS_PATH,
    'EEG_JSON_PATH': EEG_JSON_PATH,
    'TMS_JSON_PATH': TMS_JSON_PATH,
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'这些 h5/sidecar/元数据路径不存在，请先检查: {missing}')

def _decode_h5_string_list(values):
    out = []
    for item in values:
        if isinstance(item, bytes):
            out.append(item.decode('utf-8'))
        else:
            out.append(str(item))
    return out

_RAW_CACHE = {}

def load_raw():
    if 'raw' in _RAW_CACHE:
        return _RAW_CACHE['raw'].copy()
    with open(H5_MANIFEST_PATH, 'r', encoding='utf-8') as f:
        h5_meta = json.load(f)
    with h5py.File(H5_PATH, 'r') as f:
        data_uV = f['eeg_data_uV'][:]
        ch_names = _decode_h5_string_list(f['eeg_channel_names'][:])
    info = mne.create_info(ch_names=ch_names, sfreq=float(h5_meta['sampling_frequency_hz']), ch_types='eeg')
    raw = mne.io.RawArray(data_uV * 1e-6, info, verbose='ERROR')
    _RAW_CACHE['raw'] = raw
    return raw.copy()

print('所有输入路径检查通过。')
print('EEG source mode: h5')
print(f'EEG source path: {H5_PATH}')
for name, path in required_paths.items():
    print(f'{name}: {path}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

所有输入路径检查通过。
EEG source mode: h5
EEG source path: /Users/samxie/Research/EEG-Voice/ref_github/speech_decoding/exploration_outputs/edf_full_analysis/sub-P03_ses-01_task-phonemes_eeg/sub-P03_ses-01_task-phonemes_eeg_full_eeg.h5
H5_PATH: /Users/samxie/Research/EEG-Voice/ref_github/speech_decoding/exploration_outputs/edf_full_analysis/sub-P03_ses-01_task-phonemes_eeg/sub-P03_ses-01_task-phonemes_eeg_full_eeg.h5
H5_MANIFEST_PATH: /Users/samxie/Research/EEG-Voice/ref_github/speech_decoding/exploration_outputs/edf_full_analysis/sub-P03_ses-01_task-phonemes_eeg/sub-P03_ses-01_task-phonemes_eeg_full_export_manifest.json
EVENTS_PATH: /Users/samxie/Research/EEG-Voice/ref_github/speech_decoding/ds006104/sub-P03/ses-01/eeg/sub-P03_ses-01_task-phonemes_events.tsv
EVENTS_JSON_PATH: /Users/samxie/Research/EEG-Voice/ref_github/speech_decoding/ds006104/sub-P03/ses-01/eeg/sub-P03_ses-01_task-phonemes_events.json
CHANNELS_PATH: /Users/samxie/Research/EEG-Voice/ref_github/speech_decoding/ds006104/sub-P03/se

## 1. 先读元数据，不急着读整段波形

EEG 处理最好不要一上来就把整段大文件读进内存。  
先看元数据，可以快速搞清楚：

- 这是什么任务
- 采样率是多少
- 参考电极和地电极是什么
- 通道长什么样
- 事件文件包含哪些列
- TMS 的刺激参数是什么

这里对应你参考材料里的几个核心概念：

- 原始数据认知
- 通道/参考概念
- 事件与分段的关系

当前这个实验是 `2019 phoneme-pair discrimination + TMS`，也就是：  
被试听到两个音组成的刺激（例如 `a-d`、`p-u`），同时伴随不同 TMS 条件。

In [11]:
with open(EEG_JSON_PATH, 'r', encoding='utf-8') as f:
    eeg_meta = json.load(f)
with open(EVENTS_JSON_PATH, 'r', encoding='utf-8') as f:
    events_meta = json.load(f)
with open(TMS_JSON_PATH, 'r', encoding='utf-8') as f:
    tms_meta = json.load(f)

channels_df = pd.read_csv(CHANNELS_PATH, sep='\t')
events_df = pd.read_csv(EVENTS_PATH, sep='\t')

print('=== EEG JSON ===')
for key in ['TaskName', 'TaskDescription', 'Instructions', 'SamplingFrequency', 'EEGReference', 'EEGGround', 'PowerLineFrequency', 'EEGPlacementScheme', 'CapManufacturer', 'ManufacturersModelName']:
    if key in eeg_meta:
        print(f'{key}: {eeg_meta[key]}')

print('\n=== Channels ===')
print(f'通道总数: {len(channels_df)}')
print(channels_df.head())

print('\n=== Events columns ===')
print(events_df.columns.tolist())
print(f'事件总行数: {len(events_df)}')
print(events_df.head(6))

print('\n=== TMS metadata ===')
print(f"Manufacturer: {tms_meta.get('Manufacturer')}")
print(f"StimulationIntensity: {tms_meta.get('StimulationIntensity')} {tms_meta.get('StimulationIntensityType')}")
print(f"StimulationPattern: {tms_meta.get('StimulationPattern')}")
print(f"InterPulseInterval: {tms_meta.get('InterPulseInterval')} ms")
print('Targets:', ', '.join(tms_meta.get('StimulationTargets', {}).keys()))

print('\n=== events.json 字段说明预览 ===')
for key, value in list(events_meta.items())[:8]:
    print(f'{key}: {value.get("Description", "")}')

=== EEG JSON ===
TaskName: phonemes
TaskDescription: Listening to consonant-vowel and vowel-consonant phoneme pairs.
Instructions: Participants listened to audio clips immersed in white noise and responded with button presses.
SamplingFrequency: 2000
EEGReference: CPz
EEGGround: AFz
PowerLineFrequency: 60
EEGPlacementScheme: extended 10-20 system
CapManufacturer: ANT Neuro
ManufacturersModelName: eego mylab system

=== Channels ===
通道总数: 61
  name type units
0  Fp1  EEG    uV
1  Fpz  EEG    uV
2  Fp2  EEG    uV
3   F7  EEG    uV
4   F3  EEG    uV

=== Events columns ===
['onset', 'duration', 'trial_type', 'category', 'manner', 'phoneme1', 'phoneme2', 'place', 'tms_intensity', 'tms_target', 'trial', 'voicing']
事件总行数: 954
     onset  duration trial_type  category manner phoneme1 phoneme2     place  \
0  59.2610         0        TMS  alveolar   stop      NaN      NaN  alveolar   
1  59.3110         0   stimulus       NaN    NaN        a        d       NaN   
2  63.2570         0        TM

## 2. 打开 raw EEG，先感受数据长什么样

这里直接从已经导出的 `h5` 读取完整 EEG 矩阵，再先看它的头信息和基本统计。  
这样做的目的很简单：

- 先确认文件能读
- 先确认采样率、时长、通道名
- 先看一点点波形和频谱，建立直觉

这一段对应预处理的最早期观察：  
**原始波形是什么样、频谱是什么样、有没有明显的工频/漂移问题。**

In [ ]:
raw_head = load_raw()
sfreq = float(raw_head.info['sfreq'])
duration_sec = raw_head.n_times / sfreq
eeg_picks = mne.pick_types(raw_head.info, eeg=True, exclude=[])
eeg_names = [raw_head.ch_names[idx] for idx in eeg_picks]

print(raw_head)
print(f'采样率: {sfreq} Hz')
print(f'总时长: {duration_sec/60:.2f} 分钟')
print(f'EEG通道数: {len(eeg_names)}')
print(f'前10个EEG通道: {eeg_names[:10]}')

In [ ]:
preview_start = PREVIEW_START_SEC
preview_stop = PREVIEW_START_SEC + PREVIEW_DURATION_SEC
preview_picks = eeg_picks[:N_PREVIEW_CHANNELS]
preview_data, preview_times = raw_head.get_data(
    picks=preview_picks,
    start=int(preview_start * sfreq),
    stop=int(preview_stop * sfreq),
    return_times=True,
)
preview_times = preview_times + preview_start

fig, ax = plt.subplots(figsize=(14, 7))
spacing = max(float(np.nanstd(preview_data)) * 8, 100.0)
offset = 0.0
ticks, labels = [], []

for idx, pick in enumerate(preview_picks):
    ax.plot(preview_times, preview_data[idx] + offset, linewidth=0.8)
    ticks.append(offset)
    labels.append(raw_head.ch_names[pick])
    offset += spacing

ax.set_title('Raw EEG preview (first 10 seconds by default)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Channel')
ax.set_yticks(ticks)
ax.set_yticklabels(labels)
ax.grid(True, alpha=0.25)
plt.tight_layout()

if SAVE_OUTPUTS:
    fig.savefig(OUTPUT_DIR / 'raw_preview.png', dpi=220)

plt.show()

In [ ]:
raw_for_psd = load_raw()
raw_for_psd.pick('eeg')
fig = raw_for_psd.compute_psd(fmin=0.1, fmax=80, picks='eeg').plot(average=True, show=False)
fig.suptitle('Raw EEG PSD', y=1.02)
plt.show()

if SAVE_OUTPUTS:
    fig.savefig(OUTPUT_DIR / 'raw_psd.png', dpi=220, bbox_inches='tight')

## 3. 把 `events.tsv` 变成 trial 级表

`events.tsv` 这里不是“一行就是一个完整 trial”。  
在这个实验里，比较典型的结构是：

1. 一行 `TMS`
2. 下一行 `stimulus`

所以这里要显式做一件事：  
**把每个 TMS 行和它后面的 stimulus 行配对，变成真正的 trial 表。**

这一步之后，后面的 epochs / metadata 才会顺。

In [ ]:
def _clean_value(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).replace('\x00', '').strip()
    if text in {'', 'n/a', 'N/A', 'nan', 'None'}:
        return pd.NA
    return text

trial_rows = []
row_idx = 0
while row_idx < len(events_df) - 1:
    tms_row = events_df.iloc[row_idx]
    stim_row = events_df.iloc[row_idx + 1]
    if _clean_value(tms_row['trial_type']) != 'TMS' or _clean_value(stim_row['trial_type']) != 'stimulus':
        row_idx += 1
        continue

    tms_target = _clean_value(tms_row.get('tms_target'))
    trial_rows.append({
        'trial_id': int(float(tms_row['trial'])) if pd.notna(_clean_value(tms_row.get('trial'))) else row_idx // 2 + 1,
        'tms_onset_sec': float(tms_row['onset']),
        'stim_onset_sec': float(stim_row['onset']),
        'tms_to_stim_ms': (float(stim_row['onset']) - float(tms_row['onset'])) * 1000.0,
        'tms_target': tms_target,
        'condition_type': 'control' if pd.notna(tms_target) and str(tms_target).startswith('control') else 'active',
        'category': _clean_value(tms_row.get('category')),
        'place': _clean_value(tms_row.get('place')),
        'manner': _clean_value(tms_row.get('manner')),
        'voicing': _clean_value(tms_row.get('voicing')),
        'phoneme1': _clean_value(stim_row.get('phoneme1')),
        'phoneme2': _clean_value(stim_row.get('phoneme2')),
    })
    row_idx += 2

trial_df = pd.DataFrame(trial_rows)
print(f'配对后的 trial 数: {len(trial_df)}')
display(trial_df.head())

print('\nplace 计数:')
display(trial_df['place'].value_counts(dropna=False).to_frame('n'))

print('\ntms_target 计数:')
display(trial_df['tms_target'].value_counts(dropna=False).to_frame('n'))

这里是 `phonemes`，不是 `single-phoneme`。  
从 `trial_df` 里可以直接看出：

- `phoneme1` 有值
- `phoneme2` 也有值

所以每个刺激其实是**两个音组成的组合**，而不是单音。  
这也是为什么这个实验更适合看 `articulation / coarticulation`，而不是只看单音分类。

## 4. 体验式预处理：先做最小但合理的清洗

这一版只做一个**最小且合理**的流程：

1. 只保留 EEG 通道
2. notch 60 Hz
3. band-pass 0.1–40 Hz
4. average reference

为什么先这样？

- 先把 raw data 的主线流程跑通
- 先让你看到滤波和重参考前后的差异
- 避免一开始就被 ICA / 坏导插值 / 手工剔段淹没

这里也对应你参考材料里的几步：

- 滤波
- 重参考
- 先不做降采样、坏导、ICA，并说明原因

这不是说那些不重要，而是**先体验核心主线，再往上加复杂度**。

In [ ]:
raw = load_raw()
raw.pick('eeg')
raw_proc = raw.copy()

print('原始参考信息:')
print(f"EEGReference = {eeg_meta.get('EEGReference')}")
print(f"EEGGround = {eeg_meta.get('EEGGround')}")
print('\n开始做最小预处理：notch 60 -> bandpass 0.1-40 -> average reference')

raw_proc.notch_filter(freqs=[60], picks='eeg', verbose='ERROR')
raw_proc.filter(l_freq=0.1, h_freq=40.0, picks='eeg', verbose='ERROR')
raw_proc.set_eeg_reference('average', projection=False, verbose='ERROR')

print('预处理完成。')

## 5. 再看处理后的波形与频谱

这一段的重点是观察：

- 同一段 raw 波形，滤波前后有没有更平滑
- 60 Hz 附近有没有被明显压下去
- average reference 后整体波形形态有没有变化

这一步是最能建立“预处理直觉”的。

In [ ]:
compare_data_raw, compare_times = raw.get_data(
    picks=preview_picks,
    start=int(preview_start * sfreq),
    stop=int(preview_stop * sfreq),
    return_times=True,
)
compare_data_proc, _ = raw_proc.get_data(
    picks=preview_picks,
    start=int(preview_start * sfreq),
    stop=int(preview_stop * sfreq),
    return_times=True,
)
compare_times = compare_times + preview_start

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

spacing_raw = max(float(np.nanstd(compare_data_raw)) * 8, 100.0)
spacing_proc = max(float(np.nanstd(compare_data_proc)) * 8, 100.0)

offset = 0.0
ticks, labels = [], []
for idx, pick in enumerate(preview_picks):
    axes[0].plot(compare_times, compare_data_raw[idx] + offset, linewidth=0.8)
    ticks.append(offset)
    labels.append(raw.ch_names[pick])
    offset += spacing_raw
axes[0].set_title('Before preprocessing')
axes[0].set_yticks(ticks)
axes[0].set_yticklabels(labels)
axes[0].grid(True, alpha=0.25)

offset = 0.0
ticks2 = []
for idx, pick in enumerate(preview_picks):
    axes[1].plot(compare_times, compare_data_proc[idx] + offset, linewidth=0.8)
    ticks2.append(offset)
    offset += spacing_proc
axes[1].set_title('After minimal preprocessing')
axes[1].set_yticks(ticks2)
axes[1].set_yticklabels(labels)
axes[1].set_xlabel('Time (s)')
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
if SAVE_OUTPUTS:
    fig.savefig(OUTPUT_DIR / 'before_after_waveform_compare.png', dpi=220)
plt.show()

In [ ]:
fig_raw = raw.compute_psd(fmin=0.1, fmax=80, picks='eeg').plot(average=True, show=False)
fig_raw.suptitle('PSD before preprocessing', y=1.02)
plt.show()

fig_proc = raw_proc.compute_psd(fmin=0.1, fmax=80, picks='eeg').plot(average=True, show=False)
fig_proc.suptitle('PSD after preprocessing', y=1.02)
plt.show()

if SAVE_OUTPUTS:
    fig_raw.savefig(OUTPUT_DIR / 'psd_before.png', dpi=220, bbox_inches='tight')
    fig_proc.savefig(OUTPUT_DIR / 'psd_after.png', dpi=220, bbox_inches='tight')

## 6. 做 stimulus 对齐的 epochs

这里的 epoch 锚点采用 `stimulus onset`，不是 TMS onset。  
原因是这份 notebook 的重点是：**看语音刺激相关的 EEG 反应**。

窗口固定为：

- `tmin = -0.2`
- `tmax = 0.8`

基线固定为 `(None, -0.06)`，也就是到 `-60 ms` 为止。  
这样做是为了显式避开 `-50 ms` 左右的 TMS 脉冲。

这里还会做一个很简单的伪迹剔除：

- `reject = dict(eeg=150e-6)`

也就是波幅超过 `150 μV` 的 trial 直接剔掉。

In [ ]:
event_sample_idx = raw_proc.time_as_index(trial_df['stim_onset_sec'].to_numpy())
events_array = np.column_stack([
    event_sample_idx,
    np.zeros(len(trial_df), dtype=int),
    np.ones(len(trial_df), dtype=int),
]).astype(int)

epochs = mne.Epochs(
    raw_proc,
    events_array,
    event_id={'stimulus': 1},
    tmin=EPOCH_TMIN,
    tmax=EPOCH_TMAX,
    baseline=BASELINE,
    preload=True,
    metadata=trial_df.copy(),
    reject=dict(eeg=REJECT_UV * 1e-6),
    verbose='ERROR',
)

n_before = len(trial_df)
n_after = len(epochs)
drop_ratio = 1 - (n_after / n_before)
print(f'分段前 trial 数: {n_before}')
print(f'分段后保留 trial 数: {n_after}')
print(f'剔除比例: {drop_ratio:.2%}')

if drop_ratio > 0.30:
    print('警告：剔除比例偏高，后续应检查滤波、坏段、坏导和阈值设置。')

display(epochs.metadata.head())

## 7. 看最基础的 ERP 差异

这里只做两个最基础的对比，不做显著性检验：

1. `bilabial vs alveolar`
2. `control vs active TMS`

固定演示通道：`C3, Cz, C4`。  
这样做的目的不是得出最终统计结论，而是让你感受：

- EEG 里“条件平均波形”到底是什么
- 换一个条件，ERP 波形长什么样
- 数据从 raw 到 epoch 再到 evoked 的流转是什么样

In [ ]:
picks_roi = ['C3', 'Cz', 'C4']

epochs_bilabial = epochs['place == "bilabial"']
epochs_alveolar = epochs['place == "alveolar"']
epochs_control = epochs['condition_type == "control"']
epochs_active = epochs['condition_type == "active"']

evoked_bilabial = epochs_bilabial.average()
evoked_alveolar = epochs_alveolar.average()
evoked_control = epochs_control.average()
evoked_active = epochs_active.average()

def plot_roi_compare(evoked_a, evoked_b, label_a, label_b, title, save_name=None):
    evo_a = evoked_a.copy().pick(picks_roi)
    evo_b = evoked_b.copy().pick(picks_roi)
    mean_a = evo_a.data.mean(axis=0) * 1e6
    mean_b = evo_b.data.mean(axis=0) * 1e6
    times_ms = evo_a.times * 1000
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(times_ms, mean_a, label=label_a, linewidth=2)
    ax.plot(times_ms, mean_b, label=label_b, linewidth=2)
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.axhline(0, color='gray', linestyle=':', linewidth=1)
    ax.set_title(title)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude (uV)')
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    if SAVE_OUTPUTS and save_name is not None:
        fig.savefig(OUTPUT_DIR / save_name, dpi=220, bbox_inches='tight')
    plt.show()
    return fig

fig1 = plot_roi_compare(
    evoked_bilabial,
    evoked_alveolar,
    'bilabial',
    'alveolar',
    'ERP: bilabial vs alveolar (C3/Cz/C4 mean)',
    save_name='erp_bilabial_vs_alveolar.png',
)

fig2 = plot_roi_compare(
    evoked_control,
    evoked_active,
    'control',
    'active',
    'ERP: control vs active TMS (C3/Cz/C4 mean)',
    save_name='erp_control_vs_active.png',
)

## 8. 给你一个最小的数据分析视角

这一段不是正式统计分析，而是帮助你把这个实验看清楚：

- 各条件 trial 数是不是均衡
- `place × tms_target` 是怎么组合的
- `tms_to_stim_ms` 是否稳定

这里也再次强调：  
这份数据是 **phoneme pair + TMS discrimination**，不是 imagined speech 数据。

In [ ]:
summary_counts = pd.DataFrame({
    'place': trial_df['place'].value_counts(),
    'tms_target': trial_df['tms_target'].value_counts(),
}).fillna(0)
display(summary_counts)

cross_tab = pd.crosstab(trial_df['place'], trial_df['tms_target'])
display(cross_tab)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlGnBu', ax=axes[0])
axes[0].set_title('place × tms_target')
axes[0].set_xlabel('tms_target')
axes[0].set_ylabel('place')

sns.histplot(trial_df['tms_to_stim_ms'], bins=30, kde=True, ax=axes[1], color='#4c78a8')
axes[1].set_title('Distribution of TMS to stimulus interval (ms)')
axes[1].set_xlabel('ms')

plt.tight_layout()
if SAVE_OUTPUTS:
    fig.savefig(OUTPUT_DIR / 'trial_level_summary.png', dpi=220)
plt.show()

print(f"平均 TMS->stimulus 间隔: {trial_df['tms_to_stim_ms'].mean():.2f} ms")
print(f"中位数 TMS->stimulus 间隔: {trial_df['tms_to_stim_ms'].median():.2f} ms")

## 9. 保存中间结果

最后把这份 notebook 里产生的关键中间结果保存下来，方便后面继续玩数据。

默认保存：

- `trial_manifest.csv`
- `epochs-epo.fif`
- `evoked_bilabial-ave.fif`
- `evoked_alveolar-ave.fif`
- `evoked_control-ave.fif`
- `evoked_active-ave.fif`
- 若干 PNG 图
- 一个 `run_summary.json`

这一步可以让 notebook 不只是“看一眼”，而是留下后续分析能接着用的产物。

In [ ]:
if SAVE_OUTPUTS:
    trial_df.to_csv(OUTPUT_DIR / 'trial_manifest.csv', index=False)
    epochs.save(OUTPUT_DIR / 'epochs-epo.fif', overwrite=True)
    evoked_bilabial.save(OUTPUT_DIR / 'evoked_bilabial-ave.fif', overwrite=True)
    evoked_alveolar.save(OUTPUT_DIR / 'evoked_alveolar-ave.fif', overwrite=True)
    evoked_control.save(OUTPUT_DIR / 'evoked_control-ave.fif', overwrite=True)
    evoked_active.save(OUTPUT_DIR / 'evoked_active-ave.fif', overwrite=True)

run_summary = {
    'source_path': str(H5_PATH),
    'sampling_frequency_hz': float(raw.info['sfreq']),
    'raw_duration_minutes': float(raw.n_times / raw.info['sfreq'] / 60),
    'n_eeg_channels': int(len(raw.ch_names)),
    'n_event_rows': int(len(events_df)),
    'n_trials_after_pairing': int(len(trial_df)),
    'n_epochs_after_reject': int(len(epochs)),
    'drop_ratio': float(1 - len(epochs) / len(trial_df)),
    'mean_tms_to_stim_ms': float(trial_df['tms_to_stim_ms'].mean()),
    'median_tms_to_stim_ms': float(trial_df['tms_to_stim_ms'].median()),
    'baseline': list(BASELINE),
    'epoch_window_sec': [EPOCH_TMIN, EPOCH_TMAX],
    'reject_uv': float(REJECT_UV),
    'notes': [
        'This notebook intentionally uses a minimal preprocessing path.',
        'ICA and bad-channel interpolation are not included in this first-play version.',
        'Stimulus onset is used as the epoch anchor, not TMS onset.'
    ]
}

if SAVE_OUTPUTS:
    with open(OUTPUT_DIR / 'run_summary.json', 'w', encoding='utf-8') as f:
        json.dump(run_summary, f, ensure_ascii=False, indent=2)

print('保存完成。')
print(json.dumps(run_summary, ensure_ascii=False, indent=2))

## 10. 这份 notebook 暂时没做什么

为了让你先把 raw EEG 主线摸熟，这一版有意**没有**做下面几件事：

- ICA
- 坏导检测和插值
- 自动坏段检测
- 更细的 TMS artifact 处理
- 统计检验或机器学习

这些当然重要，但它们都建立在你已经搞清楚下面这些问题之后：

- 原始 EEG 文件怎么读
- events 是怎么配成 trial 的
- 预处理前后波形和频谱怎么变
- epoch / baseline / reject 到底在干什么
- evoked / ERP 是怎么从 raw 里出来的

如果你下一步想继续，我建议顺序是：

1. 先把这个 notebook 从头跑通
2. 再加一个 `resample` 版本比较前后差异
3. 再加坏导检查
4. 最后再进入 ICA